**Set environment**

In [1]:
import numpy  as np
import pandas as pd
import itertools as it
import os

In [2]:
%run ../run_config_project.py
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data


You are working with      IGVF BlueSTARR
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references



In [3]:
import fun_fasta as myfasta

## Import data

**Check file existent**

In [4]:
%env FD_RES={FD_RES}

env: FD_RES=/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results


In [5]:
%%bash
FDIRY=${FD_RES}/analysis_variant_motif_richard
ls ${FDIRY}/variant_closed_gof_bluestarr.*.fa | xargs -n 1 basename

variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.ref.fa


**Import data**

In [6]:
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fpath = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.ref.fa")

### import data
#lst_seq_ref = list(SeqIO.parse(txt_fpath_ref, "fasta"))

NUM_TOP = 10_000
lst_records = list(
    it.islice(myfasta.fun_read_fasta(txt_fpath), NUM_TOP)
)

### show
print(f"Read {len(lst_records)} records")
print("First record:")
print("Rec Idx:", lst_records[0][0])
print("Rec Seq:", lst_records[0][1])
print("Rec Ref:", lst_records[0][1][35])

Read 10000 records
First record:
Rec Idx: chr2:234143291:C:C:G
Rec Seq: AACTCAAAACCATCACCAACCGATTGCCATAGCTTCCATCATCTCCCTCCAAGGCCCATCTTAACAGGCCTCCGGATAAGCTCTCTGTTCTATTCTCATTCACCAC
Rec Ref: C


## Create pilot (larger chunk) batches for each fasta

**Helper function**

In [7]:
def chunked(iterable, size):
    itr = iter(iterable)
    while True:
        lst_chunk = list(it.islice(itr, size))
        if not lst_chunk:
            break
        yield lst_chunk

**Read and export pilot batches by chunk**

In [8]:
### output folder for pilot batches
txt_fdiry_batch = os.path.join(txt_fdiry, "batches_pilot_low_ori")
os.makedirs(txt_fdiry_batch, exist_ok=True)

### define batch sizes
num_batch_size = 100_000
num_chunk_size =   5_000

### define file name prefix
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"

In [9]:
### set input file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fpath = os.path.join(txt_fdiry, "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.ref.fa")

### read the lowest 100k records
num_total = sum(1 for _ in myfasta.fun_read_fasta(txt_fpath))
idx_start = num_total - num_batch_size
idx_end   = num_total  # idx_start + num_batch_size

lst_records = list(
    it.islice(myfasta.fun_read_fasta(txt_fpath), idx_start, idx_end)
)
print(f"Read {len(lst_records)} records for pilot batching")

### split by chunks and write
for idx_chunk, lst_chunk in enumerate(chunked(lst_records, num_chunk_size), start=1):
    ### set output file directory
    txt_fname = f"{txt_prefix}.pilot_low_chunk{idx_chunk:03d}.ref.fa.gz"
    txt_fpath = os.path.join(txt_fdiry_batch, txt_fname)

    ### write fasta records
    myfasta.fun_write_fasta(txt_fpath, lst_chunk, width=60)
    
    print(f"Wrote chunk {idx_chunk:03d}: {len(lst_chunk)} seqs -> {txt_fname}")

Read 100000 records for pilot batching
Wrote chunk 001: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk001.ref.fa.gz
Wrote chunk 002: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk002.ref.fa.gz
Wrote chunk 003: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk003.ref.fa.gz
Wrote chunk 004: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk004.ref.fa.gz
Wrote chunk 005: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk005.ref.fa.gz
Wrote chunk 006: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk006.ref.fa.gz
Wrote chunk 007: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk007.ref.fa.gz
Wrote chunk 008: 5000 seqs -> variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk008.ref.fa.gz
Wrote chunk 009: 5000 seqs -> variant_closed_gof_bluestar

**Check results**

In [10]:
%%bash
FDIRY=${FD_RES}/analysis_variant_motif_richard/batches_pilot_low_ori
ls ${FDIRY}/variant_closed_gof_bluestarr.*.fa.gz | xargs -n 1 basename

variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk001.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk002.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk003.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk004.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk005.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk006.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk007.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk008.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk009.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk010.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk011.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk012.ref.fa.gz
vari

In [11]:
%%bash
FDIRY="${FD_RES}/analysis_variant_motif_richard/batches_pilot_low_ori"
FNAME="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot_low_chunk001.ref.fa.gz"
FPATH="${FDIRY}/${FNAME}"
zcat ${FPATH} | head

>chr2:186926228:A:A:C
AATGTTTTAAAACTTTTAAGTTGGAATGAGATTTTATACTGTTTGTAGTAGAGAAACTGA
TTTAATAATTTAAATAAAAAAAAGGTTCAAATTCTGGAAAATCAAC
>chr11:107579802:A:A:T
ACCTACCATGTACCCACAAAAATTAAAAATAAAAAAAAAATCCTCCCACCTTAGCCTCCC
GAGTAACTGGGACTATAGGTGTGCATCACCATGCCCAGCTAATCTT
>chr6:95672875:A:A:G
GTGAACCTCTTACATTGCATGGCATATTTTCAGGAATAAAAAGCATGTGAATTTTTCCAG
GAAGTGCAATTTTACTACTTACTTTAGGGGGTCCATGTACCCCCCA
>chr5:27142899:C:C:G
